In [1]:
import csv
import pandas as pd
from datetime import datetime, timedelta

import warnings
warnings.filterwarnings('ignore') # Does not work for this kind of warning
pd.set_option('display.max_rows', None)


In [2]:
def validate_vacation_capacity(staff_vacations, start_date, total_days, min_required_staff=3):
    """
    Scans the calendar before generating the schedule to ensure 
    there are enough workers available every single day.
    """
    print("Checking vacation lists for staffing bottlenecks...")
    bottlenecks_found = False
    
    for day_idx in range(total_days):
        current_date = start_date + timedelta(days=day_idx)
        date_str = current_date.strftime("%Y-%m-%d")
        
        # Find out who is on vacation on this specific day
        on_vacation = [worker for worker, dates in staff_vacations.items() if date_str in dates]
        available_count = len(staff_vacations) - len(on_vacation)
        
        # If available staff drops below 3, we have a bottleneck
        if available_count < min_required_staff:
            print(f"⚠️  CRITICAL BOTTLENECK on {date_str} ({current_date.strftime('%A')}):")
            print(f"    Only {available_count} workers available! Minimum required is {min_required_staff}.")
            print(f"    Workers on vacation: {', '.join(on_vacation)}\n")
            bottlenecks_found = True
            
    if bottlenecks_found:
        print("❌ Schedule generation halted. Please adjust conflicting vacation requests above.")
        return False
    
    print("✅ No bottlenecks found! All dates have sufficient staffing levels.\n")
    return True


In [3]:
# Balance days and nights for Cezanne's calls
# Balance CNMs evenly between clinic days
def generate_fully_distributed_schedule(start_date_str, total_weeks=26):
    ft_workers = [
        "Mabel", "Reagan", "Nami", "Linda",
        "Jenn", "Dana", "Kaili", "Alison"
    ]
    pt_workers = ["Cezanne"]
    all_workers = ft_workers + pt_workers
    
    PUBLIC_HOLIDAYS = {
        "2026-10-12", "2026-11-11", "2026-11-26", "2026-12-25", "2027-01-01", "2027-01-19"
    }

    staff_vacations = {w: set() for w in all_workers}
    staff_vacations["Reagan"] = {"2026-09-24", "2026-09-25"}
    staff_vacations["Nami"] = {"2026-10-01", "2026-10-02"}
    staff_vacations["Linda"] = {"2026-12-24", "2026-12-26"}

    start_date = datetime.strptime(start_date_str, "%Y-%m-%d")
    total_days=total_weeks*7

    # Run the validation check first
    if not validate_vacation_capacity(staff_vacations, start_date, total_days):
        return  # Exit early to prevent scheduling errors or code crashes

    global_counts = {
        w: {'Day_Shifts': 0, 'Night_Shifts': 0, 'Full_Clinic': 0, 'Half_Clinic': 0, 'Total_Hours': 0} 
        for w in all_workers
    }
    deficit_hours = {w: 0.0 for w in all_workers}
    
    # NEW: Track the specific weekdays each worker has covered for hospital shifts
    # Structure: { 'Midwife Cezanne': { 'Monday': 2, 'Tuesday': 0, ... } }
    days_of_week = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday']
    worker_day_history = {w: {day: 0 for day in days_of_week} for w in all_workers}
    
    schedule_data = []
    last_night_worker = None
    start_date = datetime.strptime(start_date_str, "%Y-%m-%d")

    for week_idx in range(total_weeks):
        week_start_date = start_date + timedelta(weeks=week_idx)
        weekly_hospital_counts = {w: 0 for w in all_workers}
        week_days_manifest = {}
        
        has_holiday_this_week = False
        for day_offset in range(7):
            if (week_start_date + timedelta(days=day_offset)).strftime("%Y-%m-%d") in PUBLIC_HOLIDAYS:
                has_holiday_this_week = True
        
        weekly_base_target = 32 if has_holiday_this_week else 40

        # --- PHASE 1: Balanced Hospital Shift Allocation ---
        for day_offset in range(7):
            current_date = week_start_date + timedelta(days=day_offset)
            date_str = current_date.strftime("%Y-%m-%d")
            day_name = current_date.strftime("%A")
            
            week_days_manifest[date_str] = {
                'day_name': day_name, 'day_hospital': None, 'night_hospital': None, 
                'full_clinic_staff': [], 'half_clinic_staff': []
            }
            
            active_today = [w for w in all_workers if date_str not in staff_vacations[w]]
            
            # --- Day Shift Selection Pool ---
            available_for_day = [
                w for w in active_today 
                if w != last_night_worker and (
                    (w in ft_workers and weekly_hospital_counts[w] < 3) or 
                    (w in pt_workers and weekly_hospital_counts[w] < 1)
                )
            ]
            
            # SORTING CRITERIA:
            # 1. Shifts worked this week (asc)
            # 2. Historical count for THIS SPECIFIC DAY OF THE WEEK (asc) -> Spreads Cezanne out!
            # 3. Overall day/night shift variation imbalance balance
            # 4. Total overall hours logged
            available_for_day.sort(key=lambda w: (
                weekly_hospital_counts[w],
                worker_day_history[w][day_name], 
                global_counts[w]['Day_Shifts'] - global_counts[w]['Night_Shifts'],
                global_counts[w]['Total_Hours']
            ))
            assigned_day_worker = available_for_day[0] if available_for_day else None
            
            if assigned_day_worker:
                week_days_manifest[date_str]['day_hospital'] = assigned_day_worker
                weekly_hospital_counts[assigned_day_worker] += 1
                global_counts[assigned_day_worker]['Day_Shifts'] += 1
                global_counts[assigned_day_worker]['Total_Hours'] += 12
                worker_day_history[assigned_day_worker][day_name] += 1 # Log history

            # --- Night Shift Selection Pool ---
            available_for_night = [
                w for w in active_today 
                if w != assigned_day_worker and w != last_night_worker and (
                    (w in ft_workers and weekly_hospital_counts[w] < 3) or 
                    (w in pt_workers and weekly_hospital_counts[w] < 1)
                )
            ]
            available_for_night.sort(key=lambda w: (
                weekly_hospital_counts[w],
                worker_day_history[w][day_name],
                global_counts[w]['Night_Shifts'] - global_counts[w]['Day_Shifts'],
                global_counts[w]['Total_Hours']
            ))
            assigned_night_worker = available_for_night[0] if available_for_night else None
            
            if assigned_night_worker:
                week_days_manifest[date_str]['night_hospital'] = assigned_night_worker
                weekly_hospital_counts[assigned_night_worker] += 1
                global_counts[assigned_night_worker]['Night_Shifts'] += 1
                global_counts[assigned_night_worker]['Total_Hours'] += 12
                worker_day_history[assigned_night_worker][day_name] += 1 # Log history
            
            last_night_worker = assigned_night_worker

        # --- PHASE 2: Dynamic Clinic Deficit Backfilling ---
        for worker in all_workers:
            hospital_hours = weekly_hospital_counts[worker] * 12
            
            if worker in pt_workers:
                base_target = 12 if has_holiday_this_week else 20
                safety_max = 32
            else:
                base_target = 32 if has_holiday_this_week else 40
                safety_max = 60
                
            personal_target = base_target + deficit_hours[worker]
            needed_clinic_hours = max(0, personal_target - hospital_hours)
            
            if hospital_hours + needed_clinic_hours > safety_max:
                needed_clinic_hours = safety_max - hospital_hours
                
            req_full = int(needed_clinic_hours // 8)
            req_half = int((needed_clinic_hours % 8) // 4)
            
            full_assigned = 0
            half_assigned = 0

         # Instead of a fixed Mon->Fri loop, we sort the days dynamically for each assignment!
            def get_clinic_headcount(d_str):
                d_data = week_days_manifest[d_str]
                return len(d_data['full_clinic_staff']) + len(d_data['half_clinic_staff'])

            # Sort target dates from lowest current clinic staffing levels to highest
            sorted_dates = sorted(list(week_days_manifest.keys()), key=get_clinic_headcount)

            for date_str in sorted_dates:
                day_data = week_days_manifest[date_str]
                if full_assigned == req_full and half_assigned == req_half:
                    break

#            for date_str, day_data in week_days_manifest.items():
#                if full_assigned == req_full and half_assigned == req_half:
#                    break
                
                if day_data['day_name'] in ['Saturday', 'Sunday'] or date_str in PUBLIC_HOLIDAYS or date_str in staff_vacations[worker]:
                    continue
                
                if day_data['day_hospital'] != worker and day_data['night_hospital'] != worker:
                    prev_date_str = (datetime.strptime(date_str, "%Y-%m-%d") - timedelta(days=1)).strftime("%Y-%m-%d")
                    was_resting = prev_date_str in week_days_manifest and week_days_manifest[prev_date_str]['night_hospital'] == worker
                    
                    if not was_resting:
                        if full_assigned < req_full:
                            day_data['full_clinic_staff'].append(worker)
                            full_assigned += 1
                            global_counts[worker]['Full_Clinic'] += 1
                            global_counts[worker]['Total_Hours'] += 8
                        elif half_assigned < req_half:
                            day_data['half_clinic_staff'].append(worker)
                            half_assigned += 1
                            global_counts[worker]['Half_Clinic'] += 1
                            global_counts[worker]['Total_Hours'] += 4

            hours_worked_this_week = hospital_hours + (full_assigned * 8) + (half_assigned * 4)
            deficit_hours[worker] = max(0, personal_target - hours_worked_this_week)

        # --- PHASE 3: Compile Manifest ---
        for date_str, day_data in sorted(week_days_manifest.items()):
            is_holiday = date_str in PUBLIC_HOLIDAYS
            schedule_data.append({
                'Date': date_str,
                'Day of Week': day_data['day_name'] + (" 🎉 [HOLIDAY]" if is_holiday else ""),
                'Day Shift (12hr)': day_data['day_hospital'] if day_data['day_hospital'] else "None",
                'Night Shift (12hr)': day_data['night_hospital'] if day_data['night_hospital'] else "None",
                'Full Clinic (8hr)': "🏥 CLOSED" if is_holiday else (", ".join(day_data['full_clinic_staff']) if day_data['full_clinic_staff'] else "None"),
                'Half Clinic (4hr)': "🏥 CLOSED" if is_holiday else (", ".join(day_data['half_clinic_staff']) if day_data['half_clinic_staff'] else "None")
            })

    # Save to disk
    csv_filename = "hospital_holiday_40hr_schedule.csv"
    fields = ['Date', 'Day of Week', 'Day Shift (12hr)', 'Night Shift (12hr)', 'Full Clinic (8hr)', 'Half Clinic (4hr)']
    with open(csv_filename, mode='w', newline='') as file:
        writer = csv.DictWriter(file, fieldnames=fields)
        writer.writeheader()
        writer.writerows(schedule_data)
        
    print("6-Month Midwife Day-of-Week Distribution Audit:")
    print("-" * 55)
    for day in days_of_week:
        print(f" - {day:<12}: Cezanne scheduled {worker_day_history['Cezanne'][day]} times")
    print("6-Month Target Verification Summary:")
    print(f"{'Worker Name':<22} | {'Day Shifts':<10} | {'Night Calls':<11} | {'Full Clinic':<11} | {'Half Clinic':<11} | {'Total Hours':<12}")
    print("-" * 92)
    for worker, counts in sorted(global_counts.items()):
        print(f"{worker:<22} | {counts['Day_Shifts']:<10} | {counts['Night_Shifts']:<11} | {counts['Full_Clinic']:<11} | {counts['Half_Clinic']:<11} | {counts['Total_Hours']:<12} hrs")

generate_fully_distributed_schedule("2026-09-20")


Checking vacation lists for staffing bottlenecks...
✅ No bottlenecks found! All dates have sufficient staffing levels.

6-Month Midwife Day-of-Week Distribution Audit:
-------------------------------------------------------
 - Monday      : Cezanne scheduled 6 times
 - Tuesday     : Cezanne scheduled 6 times
 - Wednesday   : Cezanne scheduled 6 times
 - Thursday    : Cezanne scheduled 2 times
 - Friday      : Cezanne scheduled 0 times
 - Saturday    : Cezanne scheduled 0 times
 - Sunday      : Cezanne scheduled 6 times
6-Month Target Verification Summary:
Worker Name            | Day Shifts | Night Calls | Full Clinic | Half Clinic | Total Hours 
--------------------------------------------------------------------------------------------
Alison                 | 21         | 20          | 58          | 9           | 992          hrs
Cezanne                | 13         | 13          | 20          | 0           | 472          hrs
Dana                   | 20         | 22          | 56    